# Enveda CASMI 2026 — Pipeline: EDA to molecule identification

**Notebook version 1.0.0 · 2026-11-19 UTC · all project code is inline**

This notebook has no imports from `src/` and does not require any project Python files. It reads every row and column of each available Parquet file in bounded Arrow batches; it never expands the full training spectra into one in-memory pandas frame. It exports the exact first and last five raw records so you can share a compact data preview with me. Visualizations are separate, enlarged figures: static PNG at **600 DPI** plus interactive HTML with hover values.

> The visible Kaggle test file is a training-derived preview, not an independent validation set. This notebook performs EDA and unit/smoke checks only; it does not train or validate a competitive retrieval model.


## 0. Dependencies and paths

In Colab, mount Drive and set `DATA_DIR`. In local Jupyter, set it to the directory holding your downloaded `train.parquet` and `test.parquet`. The train scan is optional until that file is present; all test rows will still be processed. The notebook creates `casmi_eda_v1.0.0/` for JSON, raw endpoint previews, 600-DPI images, and hoverable HTML charts.


In [ ]:
%pip -q install numpy pandas pyarrow matplotlib seaborn plotly rdkit==2026.3.3

from pathlib import Path
from collections import Counter, deque
import json, math, random, re, sqlite3, tempfile, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display, Image, HTML

# Optional Colab Drive mount; uncomment when data are stored in Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
DATA_DIR = Path('./data')  # e.g. Path('/content/drive/MyDrive/casmi_data')
TRAIN_PATH = DATA_DIR / 'train.parquet'
TEST_PATH = DATA_DIR / 'test.parquet'
OUTPUT_DIR = Path('./casmi_pipeline_v1.0.0')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 20_000       # reduce if your machine has limited RAM
PLOT_SAMPLE_ROWS = 12_000 # plot sample only; all rows are still scanned
SEED = 42
if not TEST_PATH.is_file():
    raise FileNotFoundError(f'Missing {TEST_PATH}; set DATA_DIR to the downloaded competition data folder.')
print('Found files:', {str(p): f'{p.stat().st_size / (1024**3):.3f} GiB' for p in (TRAIN_PATH, TEST_PATH) if p.is_file()})
if not TRAIN_PATH.is_file():
    print('NOTE: train.parquet is missing. Test EDA will run, but train/test comparison and molecular-label EDA remain pending.')


## 1. Data dictionary and complete streaming scanner

Kaggle describes one row per spectrum and one prediction per `molecule_id`. Spectrum IDs should identify individual spectra; repeated molecule IDs are normal. The public data page has conflicting wording about the number of test adducts and train columns, so the actual Arrow schema below is authoritative.

The scanner computes exact full-file row, null, category, molecule/spectrum-ID and peak-array checks. A disk-backed SQLite key index avoids retaining millions of spectrum IDs in RAM. First/last raw records are retained exactly; only figures and quantiles use the bounded uniform sample.


In [ ]:
# Flatten numbers from collision-energy fields that may be scalar, list-like, or source strings such as "35HCD".
def flatten_numeric(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, np.ndarray)):
        return [number for item in value for number in flatten_numeric(item)]
    if isinstance(value, (int, float, np.number)):
        return [float(value)] if np.isfinite(value) else []
    if isinstance(value, str):
        return [float(token) for token in re.findall(r'[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?', value)]
    return []


# Convert nested Arrow/NumPy/pandas values into JSON-safe Python objects for the first/last raw rows.
def to_jsonable(value):
    if value is None or value is pd.NA:
        return None
    if isinstance(value, np.ndarray):
        return [to_jsonable(v) for v in value.tolist()]
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if isinstance(value, np.generic):
        return to_jsonable(value.item())
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, (str, int, float, bool)):
        return value
    return str(value)


# Validate one spectrum's aligned peak arrays and return counts for exact full-file QC summaries.
def inspect_peak_arrays(mzs, intensities):
    keys = ('peak_count', 'pair_length_error', 'invalid_mz_values', 'invalid_intensity_values', 'negative_intensities', 'intensity_over_one', 'base_peak_not_one')
    result = {key: 0 for key in keys}
    try:
        mz = np.asarray(mzs, dtype=np.float64)
        intensity = np.asarray(intensities, dtype=np.float64)
    except (TypeError, ValueError):
        result['pair_length_error'] = 1
        return result
    if mz.ndim != 1 or intensity.ndim != 1 or len(mz) != len(intensity):
        result['pair_length_error'] = 1
        return result
    result['peak_count'] = int(len(mz))
    result['invalid_mz_values'] = int((~np.isfinite(mz) | (mz < 0)).sum())
    result['invalid_intensity_values'] = int((~np.isfinite(intensity)).sum())
    finite = intensity[np.isfinite(intensity)]
    result['negative_intensities'] = int((finite < 0).sum())
    result['intensity_over_one'] = int((finite > 1.001).sum())
    positive = finite[finite > 0]
    if len(positive) and not np.isclose(positive.max(), 1.0, atol=0.01):
        result['base_peak_not_one'] = 1
    return result


# Scan all columns and all rows in bounded batches; preserve exact endpoints and a uniform plot sample.
def scan_parquet_complete(path, label, sample_capacity=12_000, batch_size=20_000, seed=42):
    parquet = pq.ParquetFile(path)
    schema = parquet.schema_arrow
    column_names = schema.names
    metadata_rows = int(parquet.metadata.num_rows)
    null_counts, peak_audit = Counter(), Counter({k: 0 for k in (
        'peak_count', 'pair_length_error', 'invalid_mz_values', 'invalid_intensity_values',
        'negative_intensities', 'intensity_over_one', 'base_peak_not_one',
        'num_peaks_metadata_mismatch_rows', 'num_peaks_metadata_invalid_rows')})
    categories = {name: Counter() for name in ('ingest_lib', 'adduct', 'ionization_mode', 'instrument_type', 'collision_energy_orig_units')}
    molecules = Counter()
    unique_molecule_ids = set()
    head_rows, tail_rows, sample_rows = [], deque(maxlen=5), []
    precursor_min, precursor_max = float('inf'), float('-inf')
    nonnull_spectrum_ids, rows_scanned, seen = 0, 0, 0
    rng = random.Random(seed)
    with tempfile.TemporaryDirectory(prefix=f'{label}_spectrum_ids_') as temp_dir:
        conn = sqlite3.connect(Path(temp_dir) / 'ids.sqlite')
        conn.execute('PRAGMA journal_mode=OFF')
        conn.execute('PRAGMA synchronous=OFF')
        conn.execute('CREATE TABLE ids (id TEXT PRIMARY KEY) WITHOUT ROWID')
        for batch in parquet.iter_batches(batch_size=batch_size, columns=column_names, use_threads=True):
            frame = batch.to_pandas()
            rows_scanned += len(frame)
            for col in frame.columns:
                null_counts[col] += int(frame[col].isna().sum())
            for col, counter in categories.items():
                if col in frame.columns:
                    counter.update(frame[col].fillna('<NULL>').astype(str).value_counts().to_dict())
            if 'precursor_mz' in frame:
                mass = pd.to_numeric(frame['precursor_mz'], errors='coerce').to_numpy(dtype=float)
                valid_mass = mass[np.isfinite(mass)]
                if len(valid_mass):
                    precursor_min = min(precursor_min, float(valid_mass.min()))
                    precursor_max = max(precursor_max, float(valid_mass.max()))
            records = frame.to_dict(orient='records')
            batch_ids = []
            for record in records:
                if len(head_rows) < 5:
                    head_rows.append(record)
                tail_rows.append(record)
                molecule_id = record.get('molecule_id')
                if molecule_id is not None and not pd.isna(molecule_id):
                    molecule_id = str(molecule_id)
                    unique_molecule_ids.add(molecule_id)
                    molecules[molecule_id] += 1
                spectrum_id = record.get('spectrum_id')
                if spectrum_id is not None and not pd.isna(spectrum_id):
                    batch_ids.append((str(spectrum_id),))
                    nonnull_spectrum_ids += 1
                if 'ms2_mzs' in record and 'ms2_normalized_intensities' in record:
                    peak_audit.update(inspect_peak_arrays(record['ms2_mzs'], record['ms2_normalized_intensities']))
                    if 'num_peaks' in record and pd.notna(record['num_peaks']):
                        try:
                            if int(record['num_peaks']) != len(record['ms2_mzs']):
                                peak_audit['num_peaks_metadata_mismatch_rows'] += 1
                        except (TypeError, ValueError):
                            peak_audit['num_peaks_metadata_invalid_rows'] += 1
                seen += 1
                if len(sample_rows) < sample_capacity:
                    sample_rows.append(record)
                elif sample_capacity > 0:
                    j = rng.randrange(seen)
                    if j < sample_capacity:
                        sample_rows[j] = record
            if batch_ids:
                conn.executemany('INSERT OR IGNORE INTO ids(id) VALUES (?)', batch_ids)
            conn.commit()
            del frame, records, batch_ids
        unique_spectrum_ids = int(conn.execute('SELECT COUNT(*) FROM ids').fetchone()[0])
        conn.close()
    molecules_per_count = Counter(molecules.values())
    report = {
        'label': label, 'path': str(path), 'file_bytes': path.stat().st_size,
        'parquet_metadata_rows': metadata_rows, 'rows_scanned': rows_scanned,
        'row_count_matches_metadata': rows_scanned == metadata_rows,
        'unique_molecule_ids': len(unique_molecule_ids), 'missing_molecule_id_rows': null_counts.get('molecule_id', 0),
        'molecules_with_multiple_spectra': sum(n > 1 for n in molecules.values()),
        'spectra_per_molecule_frequency': dict(molecules_per_count.most_common()),
        'spectra_per_molecule_quantiles': np.quantile(list(molecules.values()), [0, .25, .5, .75, .95, .99, 1]).tolist() if molecules else [],
        'spectrum_id_nonnull_rows': nonnull_spectrum_ids,
        'unique_spectrum_ids': unique_spectrum_ids if 'spectrum_id' in column_names else None,
        'duplicate_spectrum_id_rows': nonnull_spectrum_ids - unique_spectrum_ids if 'spectrum_id' in column_names else None,
        'columns': [{'name': field.name, 'arrow_type': str(field.type), 'nullable': field.nullable} for field in schema],
        'null_counts': dict(null_counts), 'category_counts': {k: dict(v.most_common()) for k, v in categories.items() if v},
        'peak_array_audit': dict(peak_audit),
        'precursor_mz_min_exact': precursor_min if np.isfinite(precursor_min) else None,
        'precursor_mz_max_exact': precursor_max if np.isfinite(precursor_max) else None,
        'plot_sample_rows': len(sample_rows),
        'quantiles_note': 'Precursor/peak/correlation quantiles are computed from the uniform bounded sample; row/null/category/ID/QC counts scan all rows.'}
    endpoints = {'first_5_rows': [to_jsonable(x) for x in head_rows], 'last_5_rows': [to_jsonable(x) for x in tail_rows]}
    return report, pd.DataFrame.from_records(sample_rows), endpoints


## 2. Scanner self-test and full data pass

Run the synthetic check first; it tests row-count metadata, molecule/spectrum ID aggregation, duplicate detection, and paired array QC. Then every row in each available file is scanned. The test preview can be scanned now; the train comparison will execute automatically once you place `train.parquet` beside it.


In [ ]:
# Self-test the inline scanner on a tiny local Parquet fixture before processing real files.
import pyarrow as pa
with tempfile.TemporaryDirectory() as temp_dir:
    toy = pd.DataFrame({'molecule_id':['a','a','b','b'], 'spectrum_id':['s1','s2','s2',None],
        'precursor_mz':[200.,201.,300.,301.], 'ms2_mzs':[[50.,80.],[60.],[70.,90.],[55.]],
        'ms2_normalized_intensities':[[1.,.5],[1.],[.8,.2],[1.]], 'adduct':['[M+H]+']*4,
        'num_peaks':[2,1,2,1]})
    toy_path = Path(temp_dir) / 'toy.parquet'
    pq.write_table(pa.Table.from_pandas(toy), toy_path)
    toy_report, _, toy_endpoints = scan_parquet_complete(toy_path, 'toy', sample_capacity=2, batch_size=1)
assert toy_report['rows_scanned'] == toy_report['parquet_metadata_rows'] == 4
assert toy_report['unique_molecule_ids'] == 2 and toy_report['duplicate_spectrum_id_rows'] == 1
assert toy_report['peak_array_audit']['pair_length_error'] == 0
assert len(toy_endpoints['first_5_rows']) == 4 and len(toy_endpoints['last_5_rows']) == 4
print('Inline scanner self-test passed.')

reports, samples, raw_endpoints = {}, {}, {}
started = time.time()
paths = [('train', TRAIN_PATH), ('test', TEST_PATH)]
for label, path in paths:
    if path.is_file():
        print(f'Scanning every row/column: {path} ...')
        reports[label], samples[label], raw_endpoints[label] = scan_parquet_complete(
            path, label, sample_capacity=PLOT_SAMPLE_ROWS, batch_size=BATCH_SIZE, seed=SEED)
        (OUTPUT_DIR / f'{label}_first_last_5_raw_rows.json').write_text(
            json.dumps(raw_endpoints[label], indent=2, ensure_ascii=False))
    else:
        print(f'Not present; skipped full scan: {path}')
if not reports:
    raise FileNotFoundError('No Parquet files found. Set DATA_DIR correctly.')
(OUTPUT_DIR / 'full_data_audit.json').write_text(json.dumps(reports, indent=2, ensure_ascii=False, default=str))
print(f'Full scan complete in {(time.time()-started)/60:.2f} min. Audit and raw endpoint JSON saved under {OUTPUT_DIR}.')
display(pd.DataFrame({k:{'rows scanned':v['rows_scanned'],'metadata rows':v['parquet_metadata_rows'],
    'metadata match':v['row_count_matches_metadata'],'molecules':v['unique_molecule_ids'],
    'spectra':v['unique_spectrum_ids'],'duplicate spectrum IDs':v['duplicate_spectrum_id_rows'],
    'plot sample rows':v['plot_sample_rows']} for k,v in reports.items()}).T)


## 3. Inspect actual Arrow schema, quality findings, and first/last five raw rows

The displayed preview shortens nested peak arrays for readability. The JSON endpoint files contain the complete raw first and last five records, including all fragment arrays; those compact files are suitable for sharing without sending multi-GB data. Duplicate molecule IDs are expected for multiple spectra; they are counted and interpreted separately from duplicate spectrum IDs.


In [ ]:
# Format endpoint records for a compact table while retaining complete arrays in the JSON files.
def compact_raw_row(record):
    item = {k:v for k,v in record.items() if k not in ('ms2_mzs','ms2_normalized_intensities')}
    mz = record.get('ms2_mzs') or []
    intensity = record.get('ms2_normalized_intensities') or []
    item['n_peaks'] = len(mz)
    item['mzs_first_5'] = mz[:5]
    item['intensities_first_5'] = intensity[:5]
    return item

for label, report in reports.items():
    print(f'=== {label.upper()} Arrow schema ===')
    display(pd.DataFrame(report['columns']))
    print(f'=== {label.upper()} first five rows ===')
    display(pd.DataFrame([compact_raw_row(x) for x in raw_endpoints[label]['first_5_rows']]))
    print(f'=== {label.upper()} last five rows ===')
    display(pd.DataFrame([compact_raw_row(x) for x in raw_endpoints[label]['last_5_rows']]))
    print(f'=== {label.upper()} exact missing / integrity counts ===')
    display(pd.Series(report['null_counts'], name='null rows').to_frame())
    display(pd.Series(report['peak_array_audit'], name='count').to_frame())
    print('Exact categories:', {k:pd.Series(v) for k,v in report['category_counts'].items()})


## 4. Enrich the bounded plot sample for spectrum and molecular summaries

All scan counts above are exact. Continuous distributions and correlations below are sampled for tractable plotting. Multiple spectra per molecule are repeated measurements; per-spectrum correlation is descriptive only. Per-molecule aggregates are also plotted to reduce repeated-measure bias.


In [ ]:
# Derive sample-only features for visualization; do not alter source arrays or labels.
def enrich_sample(frame):
    out = frame.copy()
    if 'ms2_mzs' in out:
        out['peak_count'] = out['ms2_mzs'].map(lambda x:len(x) if isinstance(x,(list,tuple,np.ndarray)) else 0)
    if 'ms2_normalized_intensities' in out:
        out['max_relative_intensity'] = out['ms2_normalized_intensities'].map(
            lambda x:float(np.nanmax(np.asarray(x,float))) if isinstance(x,(list,tuple,np.ndarray)) and len(x) else np.nan)
    if 'collision_energy_ev' in out:
        out['mean_collision_energy_ev'] = out['collision_energy_ev'].map(
            lambda x:float(np.nanmean(flatten_numeric(x))) if flatten_numeric(x) else np.nan)
    return out

plot_frames = {name:enrich_sample(frame) for name,frame in samples.items()}
for label, frame in plot_frames.items():
    print(label, 'sample quantiles (approximate; full counts are in audit JSON)')
    numeric = [c for c in ('precursor_mz','peak_count','base_peak_intensity','mean_collision_energy_ev') if c in frame]
    display(frame[numeric].describe(percentiles=[.25,.5,.75,.95]).T if numeric else pd.DataFrame())


## 5. Individual enlarged visualizations: 600-DPI PNG + hoverable HTML

Each visualization is its own large figure—no 3×3 subplot grid. Static charts are exported at 600 DPI; matching Plotly HTML files show exact hover values. Some plots use exact full-scan category/count summaries; distribution/correlation charts use the reservoir sample and are labeled accordingly.


In [ ]:
# Save one standalone Matplotlib figure at 600 DPI and one interactive Plotly figure with hover values.
def save_plot_pair(name, matplotlib_draw, plotly_figure, title):
    fig, ax = plt.subplots(figsize=(12,8))
    matplotlib_draw(ax)
    ax.set_title(title, fontsize=16)
    fig.tight_layout()
    png_path = OUTPUT_DIR / f'{name}.png'
    fig.savefig(png_path, dpi=600, bbox_inches='tight')
    plt.close(fig)
    html_path = OUTPUT_DIR / f'{name}.html'
    plotly_figure.write_html(html_path, include_plotlyjs='cdn', full_html=True)
    display(HTML(f'<b>{title}</b> — saved static 600-DPI PNG: {png_path}; saved interactive HTML: {html_path}'))
    # Display both artifacts inline during this same cell execution.
    display(Image(filename=str(png_path)))
    display(plotly_figure)

# Compare precursor mass distributions across the available datasets; hover shows bin counts.
frames_with_mass = {k:v for k,v in plot_frames.items() if 'precursor_mz' in v}
if frames_with_mass:
    def draw_mass(ax):
        for label, frame in frames_with_mass.items():
            sns.histplot(pd.to_numeric(frame.precursor_mz, errors='coerce').dropna(), bins=55, stat='density',
                         element='step', fill=False, ax=ax, label=label)
        ax.set_xlabel('precursor m/z'); ax.set_ylabel('density'); ax.legend()
    mass_plot = px.histogram(pd.concat([f.assign(dataset=k) for k,f in frames_with_mass.items()]),
        x='precursor_mz', color='dataset', nbins=55, barmode='overlay', opacity=.55,
        hover_data=['molecule_id','spectrum_id','adduct','ionization_mode'])
    mass_plot.update_layout(title='Precursor m/z distribution (plot sample)')
    save_plot_pair('01_precursor_mz', draw_mass, mass_plot, 'Precursor m/z — sample comparison')

# Compare peak counts; the log y axis exposes the long tail without hiding low-count spectra.
if any('peak_count' in f for f in plot_frames.values()):
    peak_frame = pd.concat([f.assign(dataset=k) for k,f in plot_frames.items() if 'peak_count' in f])
    def draw_peaks(ax):
        for label,g in peak_frame.groupby('dataset'):
            sns.histplot(g.peak_count, bins=50, element='step', fill=False, ax=ax, label=label)
        ax.set_yscale('log'); ax.set_xlabel('peaks per spectrum'); ax.set_ylabel('spectra (log)'); ax.legend()
    peak_plot = px.histogram(peak_frame, x='peak_count', color='dataset', nbins=50, opacity=.65,
        hover_data=['molecule_id','spectrum_id','precursor_mz','adduct'])
    peak_plot.update_yaxes(type='log'); peak_plot.update_layout(title='Fragments per spectrum — plot sample')
    save_plot_pair('02_peak_count', draw_peaks, peak_plot, 'Peak counts — long-tail diagnostic')

# Show exact number of molecules at each spectra-per-molecule count.
for label, report in reports.items():
    dist = pd.Series(report['spectra_per_molecule_frequency'], dtype=int).sort_index()
    if len(dist):
        display_name = f'{label}_spectra_per_molecule'
        def draw_group(ax, dist=dist):
            ax.bar(dist.index.astype(int), dist.values, color='#367c8b')
            ax.set_xlabel('spectra per molecule'); ax.set_ylabel('molecules (exact count)')
        group_plot = px.bar(x=dist.index.astype(int), y=dist.values, labels={'x':'spectra per molecule','y':'molecules'},
            hover_data={'x':dist.index.astype(int),'y':dist.values}, title=f'{label}: spectra per molecule (exact)')
        save_plot_pair(display_name, draw_group, group_plot, f'{label}: spectra per molecule — exact')

# Plot exact library-source counts when train data are present.
if 'train' in reports and 'ingest_lib' in reports['train']['category_counts']:
    source_counts = pd.Series(reports['train']['category_counts']['ingest_lib']).sort_values().tail(20)
    def draw_sources(ax):
        ax.barh(source_counts.index.astype(str), source_counts.values, color='#3b806e')
        ax.set_xscale('log'); ax.set_xlabel('spectra (log)')
    source_plot = px.bar(x=source_counts.values, y=source_counts.index.astype(str), orientation='h',
        labels={'x':'spectra (exact)','y':'ingest_lib'}, title='Training spectra by source (exact counts)',
        hover_data={'spectra':source_counts.values})
    save_plot_pair('source_library_counts', draw_sources, source_plot, 'Training library coverage')

# Show exact adduct counts by dataset.
adduct_rows=[]
for label,report in reports.items():
    for adduct,count in report['category_counts'].get('adduct',{}).items():
        adduct_rows.append({'dataset':label,'adduct':adduct,'spectra':count})
if adduct_rows:
    adduct_df=pd.DataFrame(adduct_rows)
    def draw_adduct(ax):
        sns.barplot(data=adduct_df,x='adduct',y='spectra',hue='dataset',ax=ax)
        ax.set_yscale('log'); ax.tick_params(axis='x',rotation=40)
    adduct_plot=px.bar(adduct_df,x='adduct',y='spectra',color='dataset',barmode='group',
        hover_data=['adduct','dataset','spectra'],title='Adduct counts (exact)')
    adduct_plot.update_yaxes(type='log')
    save_plot_pair('adduct_counts',draw_adduct,adduct_plot,'Adduct coverage by dataset')

# Compare collision energies; hover exposes molecule, spectrum and source metadata.
energy_frames=[]
for label,frame in plot_frames.items():
    if 'mean_collision_energy_ev' in frame:
        part=frame.dropna(subset=['mean_collision_energy_ev']).copy();part['dataset']=label
        if len(part): energy_frames.append(part)
if energy_frames:
    energy_df=pd.concat(energy_frames,ignore_index=True)
    def draw_energy(ax):
        sns.histplot(data=energy_df,x='mean_collision_energy_ev',hue='dataset',stat='density',
                     element='step',fill=False,common_norm=False,ax=ax)
        ax.set_xlabel('mean collision energy (eV)')
    energy_plot=px.histogram(energy_df,x='mean_collision_energy_ev',color='dataset',nbins=35,
        hover_data=[c for c in ('molecule_id','spectrum_id','adduct','collision_energy_orig') if c in energy_df])
    energy_plot.update_layout(title='Collision energy values (plot sample)')
    save_plot_pair('collision_energy',draw_energy,energy_plot,'Collision energy distribution')

# Visualize base peak signal against peak richness; display a descriptive Spearman coefficient.
if {'base_peak_intensity','peak_count'}.issubset(plot_frames.get('test',pd.DataFrame()).columns):
    b=plot_frames['test'].copy(); b['log_base_peak']=np.log10(pd.to_numeric(b.base_peak_intensity,errors='coerce').clip(lower=1))
    rho=b[['log_base_peak','peak_count']].corr(method='spearman').iloc[0,1]
    def draw_signal(ax):
        ax.scatter(b.log_base_peak,b.peak_count,s=14,alpha=.45)
        ax.set_xlabel('log10 raw base-peak intensity'); ax.set_ylabel('peaks per spectrum')
        ax.text(.03,.97,f'Spearman rho={rho:.3f} (sample; repeated spectra)',transform=ax.transAxes,va='top')
    signal_plot=px.scatter(b,x='log_base_peak',y='peak_count',color='ionization_mode',
        hover_data=['molecule_id','spectrum_id','precursor_mz','adduct','base_peak_intensity'],
        title=f'Base-peak intensity vs peak count — Spearman rho {rho:.3f}')
    save_plot_pair('basepeak_vs_peak_count',draw_signal,signal_plot,'Signal intensity vs spectral richness')


## 6. Correlation analysis (descriptive, not causal)

Numeric correlations are useful for diagnosing signal/coverage relationships. We compute Spearman correlations on the sampled spectra and separately on sample-level molecule aggregates; repeated spectra are not independent. The hoverable scatter plots retain molecule/spectrum IDs. Correlation should not be used as a performance claim.


In [ ]:
# Build sample-level correlation tables for each dataset and report the sample sizes explicitly.
correlation_reports={}
for label,frame in plot_frames.items():
    features=[c for c in ('precursor_mz','base_peak_intensity','peak_count','mean_collision_energy_ev') if c in frame]
    corr=frame[features].corr(method='spearman',min_periods=20) if len(features)>=2 else pd.DataFrame()
    correlation_reports[label]={'spectrum_sample_n':len(frame),'spearman':corr}
    if not corr.empty:
        display(HTML(f'<h3>{label} spectra-level Spearman correlation (sample n={len(frame)})</h3>'))
        display(corr.round(3))
        fig,ax=plt.subplots(figsize=(9,7))
        sns.heatmap(corr,vmin=-1,vmax=1,cmap='vlag',annot=True,fmt='.2f',square=True,ax=ax)
        ax.set_title(f'{label} spectrum-sample Spearman correlation (n={len(frame)})')
        fig.tight_layout();fig.savefig(OUTPUT_DIR/f'{label}_correlation_heatmap.png',dpi=600,bbox_inches='tight');plt.close(fig)
        html=px.imshow(corr,zmin=-1,zmax=1,color_continuous_scale='RdBu_r',text_auto='.2f',
            title=f'{label} sample Spearman correlations; hover shows feature pair/value')
        html.write_html(OUTPUT_DIR/f'{label}_correlation_heatmap.html',include_plotlyjs='cdn')
    if 'molecule_id' in frame and 'peak_count' in frame:
        group_cols=[c for c in ('precursor_mz','peak_count','base_peak_intensity','mean_collision_energy_ev') if c in frame]
        molecule_level=frame.groupby('molecule_id',dropna=True)[group_cols].median(numeric_only=True)
        if len(molecule_level)>=20 and len(group_cols)>=2:
            molecule_corr=molecule_level.corr(method='spearman',min_periods=20)
            correlation_reports[label]['molecule_sample_n']=len(molecule_level)
            correlation_reports[label]['molecule_level_spearman']=molecule_corr
            display(HTML(f'<h3>{label} molecule-aggregated Spearman correlation (sample molecules n={len(molecule_level)})</h3>'))
            display(molecule_corr.round(3))
            fig,ax=plt.subplots(figsize=(9,7));sns.heatmap(molecule_corr,vmin=-1,vmax=1,cmap='vlag',annot=True,fmt='.2f',square=True,ax=ax)
            ax.set_title(f'{label} molecule-aggregated correlation (sample)')
            fig.tight_layout();fig.savefig(OUTPUT_DIR/f'{label}_molecule_correlation_600dpi.png',dpi=600,bbox_inches='tight');plt.close(fig)
(OUTPUT_DIR/'sample_correlations.json').write_text(json.dumps({k:{key:(value.to_dict() if isinstance(value,pd.DataFrame) else value) for key,value in item.items()} for k,item in correlation_reports.items()},indent=2,default=str))


### Interpretation of the supplied 12,000-row train sample and 1,213-row test preview

The supplied spectra-level Spearman tables show peak count and raw base-peak intensity are positively associated in both samples (train ρ=0.560; test ρ=0.616). Precursor m/z has only weak association with peak count (train −0.091; test 0.086) and with base-peak intensity (train −0.242; test 0.043). Mean collision energy has a modest positive association with peak count in train (ρ=0.278), but little association in test (−0.064); this may reflect the mixed source/instrument composition of train and the single-instrument test domain. These are descriptive sample correlations, not causal effects or predictive performance.

For the 400 test-preview molecules, molecule-aggregated peak-count/base-peak correlation remains positive (ρ=0.586), while energy correlations are near zero. Aggregation reduces repeated-spectrum influence but does not make this training-derived preview independent validation.

Train/test sample quantiles differ substantially: train median peak count 42 versus test 230; train median precursor m/z ≈350.2 versus test ≈329.2. Treat these as sample summaries (train n=12,000; test n=1,213), not full-population contrasts. The train sample also has only 6,517 non-null raw base-peak values and 10,420 non-null mean-energy values, versus all 1,213 test rows. Comparisons involving these fields are therefore subject to missingness and source effects.


## 7. Molecular structure statistics (training only)

Test structures are unknown. If training data are present, summarize the formula and SMILES labels without treating the test preview as labeled. RDKit is pinned to the official competition version. These are descriptive molecular-composition statistics, not a model.


In [ ]:
if 'train' in samples and 'normalized_smiles' in samples['train']:
    from rdkit import Chem, rdBase
    print('RDKit:',rdBase.rdkitVersion)
    train_sample=samples['train'].dropna(subset=['normalized_smiles']).copy()
    train_sample['smiles_length']=train_sample.normalized_smiles.astype(str).str.len()
    train_sample['valid_smiles']=train_sample.normalized_smiles.map(lambda s:Chem.MolFromSmiles(str(s)) is not None)
    display(train_sample[['smiles_length','valid_smiles']].describe(include='all'))
    if 'molecular_formula' in train_sample:
        display(train_sample.molecular_formula.value_counts().head(30).to_frame('sample spectra'))
    fig,ax=plt.subplots(figsize=(12,8));sns.histplot(train_sample.smiles_length,bins=50,ax=ax)
    ax.set_title('Training SMILES string length (plot sample)');ax.set_xlabel('SMILES characters')
    fig.tight_layout();fig.savefig(OUTPUT_DIR/'training_smiles_length_600dpi.png',dpi=600,bbox_inches='tight');plt.close(fig)
else:
    print('Training molecular statistics unavailable: train.parquet with normalized_smiles is not present.')


### Molecular-label sample interpretation

In the supplied 12,000-spectrum train sample, RDKit 2026.09.2 parsed all 12,000 SMILES successfully. The mean SMILES length was 44.4 characters (median 40; range 7–303). Formula counts shown above are spectrum-level counts and repeated structures contribute multiple times: for example, C19H23N3O3 occurs 24 times, and C19H24N4O2 20 times in this sample. Do not interpret these as unique-molecule prevalence; use `inchikey14`/canonical identity grouping for structure-level counts. Test labels are unavailable, so no test structure statistics can be computed.

This is a bounded sample description rather than a full-train estimate. The broader train set mixes sources and instruments; the test domain is timsTOF and differs chemically from the largest synthetic drug-like training source, so validation should be molecule-grouped and domain-aware.

## 8. Task 2 — Molecular identity and official MRR@25 smoke tests

The test file has no answers, so this is a code-level unit test against known examples from Kaggle's public metric. It is not a validation score.


In [ ]:
# Convert a SMILES to the competition InChIKey14 after stripping stereo before tautomer canonicalization.
def canonicalize_smiles_to_ik14(smiles):
    from rdkit import Chem, rdBase
    from rdkit.Chem.MolStandardize import rdMolStandardize
    accepted={'2026.03.3','2026.3.3'}
    if rdBase.rdkitVersion not in accepted:
        raise RuntimeError(f'Expected competition RDKit 2026.03.3; got {rdBase.rdkitVersion}')
    if not isinstance(smiles,str) or not smiles.strip(): return None
    try:
        mol=Chem.MolFromSmiles(smiles)
        if mol is None: return None
        Chem.RemoveStereochemistry(mol)
        canon=rdMolStandardize.TautomerEnumerator().Canonicalize(mol)
        return Chem.MolToInchiKey(canon)[:14]
    except (ValueError,RuntimeError,TypeError):
        return None


# Compute first-hit reciprocal rank across one row per molecule; predictions are semicolon-separated.
def mrr_at_25(solution, submission):
    if solution.molecule_id.duplicated().any() or submission.molecule_id.duplicated().any():
        raise ValueError('Expected one row per molecule_id')
    if set(solution.molecule_id)!=set(submission.molecule_id):
        raise ValueError('Solution/submission molecule IDs differ')
    targets=dict(zip(solution.molecule_id,solution.smiles.map(canonicalize_smiles_to_ik14)))
    scores=[]
    for row in submission.itertuples(index=False):
        score=0.0
        for rank,candidate in enumerate([x.strip() for x in row.smiles.split(';') if x.strip()][:25],1):
            if canonicalize_smiles_to_ik14(candidate)==targets[row.molecule_id]:
                score=1.0/rank;break
        scores.append(score)
    return float(np.mean(scores)) if scores else 0.0


# Deduplicate a ranked list by competition identity while retaining first representative SMILES.
def deduplicate_by_ik14(candidates,max_rank=25):
    unique=[];seen=set()
    for smiles in candidates:
        key=canonicalize_smiles_to_ik14(smiles)
        if key and key not in seen:
            seen.add(key);unique.append(smiles)
        if len(unique)>=max_rank:break
    return unique

# Public metric example: rank-1 and rank-2 hits produce MRR 0.75.
solution=pd.DataFrame({'molecule_id':['m1','m2'],'smiles':['CCO','c1ccccc1']})
submission=pd.DataFrame({'molecule_id':['m1','m2'],'smiles':['CCO;CC(=O)O','CC(=O)O;c1ccccc1']})
assert abs(mrr_at_25(solution,submission)-0.75)<1e-12
assert canonicalize_smiles_to_ik14('CC(=O)C')==canonicalize_smiles_to_ik14('C=C(O)C')
assert canonicalize_smiles_to_ik14('C[C@H](N)C(=O)O')==canonicalize_smiles_to_ik14('CC(N)C(=O)O')
assert len(deduplicate_by_ik14(['CCO','OCC','CC(=O)O']))==2
print('Task 2 identity/MRR smoke tests passed. No score is available for hidden data.')


## 9. Task 3 — Deterministic preprocessing and adduct smoke checks

This cell tests reproducible array validation, sorting, optional relative-intensity filtering and neutral-mass conversion for the ten adducts documented for test. It does not select noise thresholds or claim a validated preprocessing improvement.


In [ ]:
# Convert singly charged adduct m/z to neutral mass; unknown adduct labels return NaN, never raw m/z.
def neutral_mass_from_adduct(mz,adduct):
    shifts={'[m+h]+':1.007276,'[m+nh4]+':18.033823,'[m-h2o+h]+':-17.003289,
        '[m-2h2o+h]+':-35.013854,'[m+na]+':22.989218,'[m+k]+':38.963158,
        '[m-h]-':-1.007276,'[m-h2o-h]-':-19.017841,'[m+ch2o2-h]-':44.998201,'[m+cl]-':34.969402}
    try: observed=float(mz)
    except (TypeError,ValueError): return float('nan')
    shift=shifts.get(str(adduct).replace(' ','').lower())
    return observed-shift if shift is not None and np.isfinite(observed) else float('nan')


# Deterministically validate/transform aligned peak lists; caller controls a relative-intensity floor.
def preprocess_spectrum(mzs,intensities,precursor_mz=None,relative_floor=0.0):
    mz=np.asarray(mzs,dtype=float);inten=np.asarray(intensities,dtype=float)
    if mz.ndim!=1 or inten.ndim!=1 or len(mz)!=len(inten):
        raise ValueError('m/z and intensity arrays must be aligned 1-D arrays')
    valid=np.isfinite(mz)&np.isfinite(inten)&(mz>=0)&(inten>=0)
    if precursor_mz is not None and np.isfinite(precursor_mz): valid &= mz<=float(precursor_mz)+2.0
    mz,inten=mz[valid],inten[valid]
    if len(inten) and inten.max()>0:
        inten=inten/inten.max()
        keep=inten>=relative_floor
        mz,inten=mz[keep],inten[keep]
    order=np.argsort(mz,kind='stable')
    return mz[order],inten[order]

# Unit smoke checks cover deterministic output, invalid peaks, length mismatch and all documented test adduct shifts.
raw_mz=[100.0,50.0,float('nan'),400.0];raw_int=[.5,1.0,.2,.1]
a=preprocess_spectrum(raw_mz,raw_int,precursor_mz=200,relative_floor=.05)
b=preprocess_spectrum(raw_mz,raw_int,precursor_mz=200,relative_floor=.05)
assert np.array_equal(a[0],b[0]) and np.array_equal(a[1],b[1]) and a[0].tolist()==[50.0,100.0]
assert np.isfinite(neutral_mass_from_adduct(301.007276,'[M+H]+'))
for adduct,shift in {'[M+H]+':1.007276,'[M+NH4]+':18.033823,'[M-H2O+H]+':-17.003289,
 '[M-2H2O+H]+':-35.013854,'[M+Na]+':22.989218,'[M+K]+':38.963158,
 '[M-H]-':-1.007276,'[M-H2O-H]-':-19.017841,'[M+CH2O2-H]-':44.998201,'[M+Cl]-':34.969402}.items():
 assert abs(neutral_mass_from_adduct(300+shift,adduct)-300)<1e-5,adduct
try: preprocess_spectrum([1,2],[1])
except ValueError: pass
else: raise AssertionError('Mismatched arrays were not rejected')
print('Task 3 deterministic preprocessing/adduct smoke tests passed; no tuning claim is made.')


## 10. Expected deliverables, task breakdown, and acceptance gates

### Task 1 — Data understanding / EDA (P0)
**Objective:** Understand actual train/test schema, data quality, relationships, and acquisition/source coverage before modeling.
- Load both Parquet files and inspect Arrow schema, IDs, nulls, duplicate spectrum IDs, peak-array integrity, precursor/adduct/energy fields and train molecular labels.
- Determine spectra per molecule, distributions by source/instrument/adduct/mode/energy and train/test domain differences.
- Deliver: schema report, missing/duplicate analysis, spectrum/peak stats, molecular structure stats (train only), train/test comparison, data dictionary, raw endpoint JSON, 600-DPI figures and hoverable HTML.
- Acceptance: all available files scanned to completion; `rows_scanned` matches Parquet metadata; actual schema and documentation discrepancies resolved from local files; duplicate/missing IDs and array integrity reviewed; spectra/molecule and source/adduct/instrument/energy coverage interpreted; audit JSON/plots saved. No model conclusion from the training-derived preview.

### After EDA
**Task 2 — Identity/MRR@25 (P0):** RDKit molecules, pinned tautomer canonicalization, InChIKey14, deduplication, official MRR tests. Gate: reproduce public scorer examples and edge cases.

**Task 3 — Preprocessing (P1):** aligned peak validation, deterministic transforms, adduct/charge/mode/energy handling. Gate: deterministic tests and empirically justified filtering.

**Task 4 — Spectral library baseline (P0):** exact/modified cosine, entropy, precursor filtering, multi-spectrum evidence fusion. Gate: leakage-safe molecule-held-out Recall@K and MRR@25.

**Task 5 — Candidate generation (P1):** neutral mass/formula, training/public permitted structures, identity dedupe. Gate: held-out candidate recall, coverage and licensing.

**Task 6 — Deep learning (P2):** spectrum-to-fingerprint/embedding/contrastive models after baseline. Gate: improves molecule-disjoint validation under feasible memory/time.

**Task 7 — Ranking/ensemble (P2):** spectral, mass, formula, fingerprint and fragmentation evidence. Gate: reproducible MRR gain over strongest channel.

**Task 8 — Offline submission (P0):** hidden IDs, valid unique SMILES, <=25 candidates, offline <=9h. Gate: actual end-to-end rerun and scorer validation.

**Current scope:** this notebook implements data audit, identity/MRR smoke tests, and deterministic preprocessing/adduct smoke tests. It does not complete Tasks 4–8 or claim a competitive model. The test preview is not independent validation; all predictive claims require identity-disjoint validation and actual train labels.


# Enveda CASMI 2026 Pipeline — Tasks 2–8

This version preserves the v0.9.0 EDA and adds a staged, auditable path from molecular identity through retrieval, candidate ranking, optional learning, and submission generation. It is a **pipeline workbench**, not a claimed competitive solution. The local checkout currently has `test.parquet` but not `train.parquet`; benchmark and learned-model stages therefore remain gated until training labels are mounted. Never score on the training-derived visible test preview as if it were independent validation.

**Competition constraints:** one ranked candidate list per `molecule_id`, max 25 SMILES; offline Kaggle notebook, <=9 hours. RDKit identity must match the published scorer.


## Task 2 — Official molecular identity and MRR@25 (P0)

The identity key removes stereochemistry, canonicalizes tautomers with the pinned RDKit tautomer enumerator, then compares the first 14 InChIKey characters. Use this same key to deduplicate candidate lists. The cell below tests public-style connectivity/tautomer/stereo examples and MRR positions; RDKit version mismatch is an explicit stop, not a silent fallback.


In [ ]:
# Inline competition identity implementation; install cell above pins RDKit 2026.03.3.
from functools import lru_cache
from rdkit import Chem, rdBase, RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem.MolStandardize import rdMolStandardize
EXPECTED_RDKIT = {'2026.03.3', '2026.3.3'}
print('RDKit runtime:', rdBase.rdkitVersion)

def ik14(smiles):
    if not isinstance(smiles, str) or not smiles.strip(): return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    Chem.RemoveStereochemistry(mol)
    canonical = rdMolStandardize.TautomerEnumerator().Canonicalize(mol)
    key = Chem.MolToInchiKey(canonical)
    return key[:14] if key else None

def dedup_smiles(candidates, k=25):
    seen, kept = set(), []
    for s in candidates:
        key = ik14(s)
        if key and key not in seen:
            kept.append(s.strip()); seen.add(key)
        if len(kept) >= k: break
    return kept

def mrr25(solution, ranked):
    # solution/ranked are dict[molecule_id, SMILES or semicolon-delimited SMILES]
    if set(solution) != set(ranked): raise ValueError('solution and prediction IDs must match exactly')
    scores=[]
    for mid, truth in solution.items():
        target=ik14(truth); guesses=ranked[mid].split(';') if isinstance(ranked[mid],str) else ranked[mid]
        score=0.
        for rank, guess in enumerate(dedup_smiles(guesses,25),1):
            if ik14(guess)==target: score=1./rank; break
        scores.append(score)
    return float(np.mean(scores)) if scores else 0.

# Public metric example: glucose tautomer/stereo-insensitive identity, rank reciprocal values.
assert ik14('OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O') == ik14('OCC1OC(O)C(O)C(O)C1O')
assert ik14('CCO') == ik14('OCC')
assert ik14('not a smiles') is None
assert len(dedup_smiles(['CCO','OCC','CCN'])) == 2
assert abs(mrr25({'m1':'CCO','m2':'CCN'}, {'m1':['CCN','CCO'],'m2':['CCN']})-.75)<1e-12
assert abs(mrr25({'m1':'CCO'}, {'m1':['CCN','CCO']})-.5)<1e-12
assert abs(mrr25({'m1':'CCO'}, {'m1':['C'*i for i in range(1,25)]+['CCO']})-.04)<1e-12
assert mrr25({'m1':'CCO'}, {'m1':['CCN']})==0.
if rdBase.rdkitVersion not in EXPECTED_RDKIT:
    print('IDENTITY GATE BLOCKED: install the official RDKit 2026.03.3 wheel before claiming scorer parity.')
else: print('Task 2 public-style identity/MRR assertions passed.')


## Task 3 — Deterministic preprocessing (P1)

Validate the aligned arrays before transforming; retain zero/low peaks by default (filter thresholds are tunable, not yet empirically justified). Return sorted, merged m/z bins, normalized intensities, and explicit diagnostics. Energy is carried as a list in eV when supplied; no silent eV/NCE conversion is done in this cell. Adduct-to-neutral-mass conversion supports the competition's singly charged adducts.


In [ ]:
# Deterministic peak sanitation and known singly charged test-adduct neutral-mass shifts.
ADDUCT_SHIFT = {'[M+H]+':1.007276466621,'[M+NH4]+':18.033823,'[M-H2O+H]+':-17.003289,
 '[M-2H2O+H]+':1.007276466621-2*18.010564684,'[M+Na]+':22.989220702,'[M+K]+':38.9631581,
 '[M-H]-':-1.007276466621,'[M-H2O-H]-':-19.01784115,'[M+CH2O2-H]-':44.998201,'[M+Cl]-':34.969402}

def preprocess_spectrum(mzs, intensities, precursor_mz=None, min_relative=0.0, max_peaks=None, bin_tolerance=0.0):
    mz=np.asarray(mzs,dtype=np.float64); it=np.asarray(intensities,dtype=np.float64)
    if mz.ndim!=1 or it.ndim!=1 or len(mz)!=len(it): raise ValueError('m/z and intensity arrays must be aligned 1D arrays')
    good=np.isfinite(mz)&np.isfinite(it)&(mz>0)&(it>=0)
    if precursor_mz is not None and np.isfinite(precursor_mz): good &= mz <= float(precursor_mz)+2.0
    mz,it=mz[good],it[good]
    if not len(mz) or it.max()<=0: return np.array([],float),np.array([],float)
    it=it/it.max(); keep=it>=min_relative; mz,it=mz[keep],it[keep]
    if max_peaks and len(mz)>max_peaks:
        idx=np.argsort(-it,kind='stable')[:max_peaks]; mz,it=mz[idx],it[idx]
    order=np.argsort(mz,kind='stable'); mz,it=mz[order],it[order]
    if bin_tolerance>0 and len(mz):
        out_mz=[];out_i=[];start=0
        for j in range(1,len(mz)+1):
            if j==len(mz) or mz[j]-mz[j-1]>bin_tolerance:
                weights=it[start:j];out_mz.append(float(np.average(mz[start:j],weights=weights)));out_i.append(float(weights.sum()));start=j
        mz,it=np.array(out_mz),np.array(out_i)
        it=it/it.max()
    return mz,it

def validate_spectrum_metadata(adduct, ionization_mode, collision_energy_ev=None):
    expected_mode='positive' if isinstance(adduct,str) and adduct.endswith('+') else 'negative' if isinstance(adduct,str) and adduct.endswith('-') else None
    mode_ok=ionization_mode in {'positive','negative'} and (expected_mode is None or ionization_mode==expected_mode)
    energies=[] if collision_energy_ev is None else np.asarray(collision_energy_ev,dtype=float).reshape(-1).tolist()
    energies_ok=all(np.isfinite(x) and x>=0 for x in energies)
    return {'known_test_adduct':adduct in ADDUCT_SHIFT,'mode_consistent':bool(mode_ok),'energies_ev':energies,'energies_valid':bool(energies_ok)}

def neutral_mass(precursor_mz, adduct):
    shift=ADDUCT_SHIFT.get(adduct)
    return float(precursor_mz)-shift if shift is not None and np.isfinite(precursor_mz) else np.nan

# Repeatability, alignment rejection, precursor filter, normalization and representative adduct tests.
t1=preprocess_spectrum([50,75,100,300],[.2,1,.5,.1],precursor_mz=200,max_peaks=10)
t2=preprocess_spectrum([50,75,100,300],[.2,1,.5,.1],precursor_mz=200,max_peaks=10)
assert np.array_equal(t1[0],t2[0]) and np.array_equal(t1[1],t2[1]) and t1[1].max()==1 and 300 not in t1[0]
try: preprocess_spectrum([1,2],[1])
except ValueError: pass
else: raise AssertionError('misaligned arrays must fail')
assert abs(neutral_mass(101.007276466621,'[M+H]+')-100)<1e-8
assert validate_spectrum_metadata('[M-H]-','negative',[20,40])['energies_valid']
assert not validate_spectrum_metadata('[M-H]-','positive',[20])['mode_consistent']
print('Task 3 deterministic preprocessing/adduct assertions passed; thresholds remain to be validated empirically.')


## Task 4 — Spectral library retrieval baseline (P0)

Implement sparse peak-vector cosine, modified cosine (neutral-loss-compatible peak matching), entropy similarity, precursor compatibility, and multi-spectrum molecule-level fusion. This first, transparent implementation prioritizes auditability; tune tolerance, transforms, source weights, and fusion only on molecule-held-out validation. Exact full-library indexing can be expensive, so `MAX_TRAIN_ROWS` is a reproducible development cap; set it to `None` only after measuring runtime/RAM.


In [ ]:
# Pairwise spectral similarities. Inputs are preprocessed (m/z, normalized intensity) tuples.
def _cosine_pairs(a,b,tol=0.02,shift=0.0):
    am,ai=a; bm,bi=b
    if len(am)==0 or len(bm)==0:return 0.
    score=0.; j0=0
    for x,w in zip(am,ai):
        target=x+shift
        while j0<len(bm) and bm[j0]<target-tol:j0+=1
        for j in range(j0,min(len(bm),j0+4)):
            if abs(bm[j]-target)<=tol: score+=w*bi[j]; break
    return float(score/(np.linalg.norm(ai)*np.linalg.norm(bi)+1e-12))

def _entropy_similarity(a,b,tol=0.02,shift=0.0):
    am,ai=a; bm,bi=b
    if len(am)==0 or len(bm)==0:return 0.
    # Build aligned intensity distributions over matched peak pairs plus unmatched peaks.
    used=set(); pa=[]; pb=[]
    for x,w in zip(am,ai):
        choices=[j for j,y in enumerate(bm) if j not in used and abs(y-(x+shift))<=tol]
        if choices:
            j=min(choices,key=lambda j:abs(bm[j]-(x+shift)));used.add(j);pa.append(float(w));pb.append(float(bi[j]))
        else:pa.append(float(w));pb.append(0.)
    for j,w in enumerate(bi):
        if j not in used:pa.append(0.);pb.append(float(w))
    pa=np.asarray(pa);pb=np.asarray(pb);pa/=pa.sum()+1e-12;pb/=pb.sum()+1e-12
    mid=(pa+pb)/2
    def entropy(x):
        x=x[x>0];return float(-np.sum(x*np.log2(x)))
    jsd=max(0.,entropy(mid)-.5*entropy(pa)-.5*entropy(pb))
    return float(np.clip(1.-jsd,0.,1.))

def spectral_similarity(a,b,tol=0.02,precursor_delta=None):
    exact=_cosine_pairs(a,b,tol,0.)
    modified=exact; entropy=_entropy_similarity(a,b,tol,0.)
    if precursor_delta is not None and np.isfinite(precursor_delta):
        modified=max(exact,_cosine_pairs(a,b,tol,float(precursor_delta)))
        entropy=max(entropy,_entropy_similarity(a,b,tol,float(precursor_delta)))
    return {'cosine':exact,'modified_cosine':modified,'entropy_similarity':entropy}

def fuse_spectrum_evidence(query_spectra, library_by_identity, tol=0.02):
    # Max + mean evidence across query spectra; aggregate candidate scores per identity, not per spectrum.
    result={}
    for identity,entry in library_by_identity.items():
        vals=[]
        for q in query_spectra:
            for ref in entry['spectra']:
                delta=(q.get('precursor_mz',np.nan)-ref.get('precursor_mz',np.nan))
                vals.append(spectral_similarity(q['peaks'],ref['peaks'],tol,delta).get('modified_cosine',0.))
        if vals: result[identity]={'max':max(vals),'mean':float(np.mean(vals)),'score':.7*max(vals)+.3*float(np.mean(vals))}
    return sorted(result.items(),key=lambda x:(-x[1]['score'],x[0]))

# Similarity sanity: identical wins, zero-overlap is zero.
sp=((np.array([50.,75.,100.]),np.array([.2,1.,.4])),(np.array([50.,75.,100.]),np.array([.2,1.,.4])))
assert spectral_similarity(sp[0],sp[1])['cosine'] > .999
assert _cosine_pairs(sp[0],(np.array([500.]),np.array([1.])))==0.
print('Task 4 similarity primitive checks passed; benchmark gate awaits train labels.')


## Task 5 — Candidate pool and molecule-disjoint validation (P1)

Build the structure pool from labeled training SMILES, deduplicated by official InChIKey14. Neutral-mass candidates are retrieved within a configurable ppm window; formula-composition filtering is optional if formula labels are present. Public databases must be supplied as a separately audited, permitted local structure file—this notebook does not download or presume licensing. Split identities before creating any retrieval index.


In [ ]:
# Deterministic identity-group split and mass candidate recall utilities.
import hashlib
def identity_split(key, valid_fraction=.2, seed='casmi26'):
    h=hashlib.sha256((seed+str(key)).encode()).digest()
    return int.from_bytes(h[:8],'big')/2**64 < valid_fraction

def split_by_identity(frame, valid_fraction=.2, seed='casmi26'):
    keys=frame.normalized_smiles.map(ik14)
    is_valid=keys.map(lambda k: identity_split(k,valid_fraction,seed) if k else False)
    return frame.loc[~is_valid].copy(),frame.loc[is_valid].copy()

def make_candidate_pool(train_df):
    rows={}
    for r in train_df.itertuples(index=False):
        d=r._asdict(); smi=d.get('normalized_smiles'); ident=ik14(smi)
        if not ident: continue
        if ident not in rows: rows[ident]={'smiles':smi,'spectra':[],'formula':d.get('molecular_formula')}
        try: peaks=preprocess_spectrum(d['ms2_mzs'],d['ms2_normalized_intensities'],d.get('precursor_mz'))
        except (ValueError,KeyError): continue
        rows[ident]['spectra'].append({'peaks':peaks,'precursor_mz':d.get('precursor_mz'),'adduct':d.get('adduct')})
    return rows

def mass_candidates(query_mz, query_adduct, pool, ppm=10.):
    qmass=neutral_mass(query_mz,query_adduct)
    if not np.isfinite(qmass):return []
    # Formula-derived monoisotopic masses for candidate structures.
    out=[]
    for ident,entry in pool.items():
        mol=Chem.MolFromSmiles(entry['smiles'])
        if mol is None:continue
        from rdkit.Chem import rdMolDescriptors
        # RDKit exact mass of neutral candidate; avoid using source precursor m/z as neutral mass.
        mass=rdMolDescriptors.CalcExactMolWt(mol)
        err=1e6*abs(mass-qmass)/qmass
        if err<=ppm:out.append((ident,entry['smiles'],float(err)))
    return sorted(out,key=lambda x:(x[2],x[0]))


In [ ]:
# Score held-out query molecules against an index built ONLY from training identities.
def evaluate_retrieval(train_df, valid_df, max_rows=None, ppm=20., tol=.02):
    # Caller must pass identity-disjoint frames; assert before making any index.
    train_ids={ik14(x) for x in train_df.normalized_smiles.dropna().unique()}
    valid_ids={ik14(x) for x in valid_df.normalized_smiles.dropna().unique()}
    overlap=train_ids & valid_ids
    if overlap: raise ValueError(f'identity leakage: {len(overlap)} identities overlap')
    if max_rows: train_df=train_df.iloc[:max_rows]
    pool=make_candidate_pool(train_df); hits=0; reciprocal=[]; evaluated=0; covered=0
    for ident,g in valid_df.groupby(valid_df.normalized_smiles.map(ik14)):
        if not ident or len(g)==0:continue
        query=[]
        for _,r in g.iterrows():
            try: query.append({'peaks':preprocess_spectrum(r.ms2_mzs,r.ms2_normalized_intensities,r.precursor_mz),'precursor_mz':r.precursor_mz})
            except ValueError: pass
        candidates=mass_candidates(float(g.iloc[0].precursor_mz),str(g.iloc[0].adduct),pool,ppm)
        covered += int(any(cid==ident for cid,_,_ in candidates))
        scored=[]
        for cid,smiles,mass_error in candidates:
            entry=pool[cid]; vals=[]
            for q in query:
                for ref in entry['spectra']:
                    delta=q['precursor_mz']-ref['precursor_mz'] if np.isfinite(ref['precursor_mz']) else None
                    vals.append(spectral_similarity(q['peaks'],ref['peaks'],tol,delta)['modified_cosine'])
            spectral=.7*max(vals)+.3*float(np.mean(vals)) if vals else 0.
            mass_score=max(0.,1.-mass_error/ppm)
            scored.append((cid,.85*spectral+.15*mass_score))
        ranked=[cid for cid,_ in sorted(scored,key=lambda z:(-z[1],z[0]))[:25]]
        pos=ranked.index(ident)+1 if ident in ranked else 0
        hits+=int(pos>0);reciprocal.append(1/pos if pos else 0.);evaluated+=1
    return {'molecules':evaluated,'candidate_recall_at_25':hits/max(evaluated,1),
            'mass_candidate_coverage':covered/max(evaluated,1),'mrr_at_25':float(np.mean(reciprocal)) if reciprocal else 0.,'pool_identities':len(pool)}

print('Group split utility and held-out evaluator defined. Train parquet absent means metrics are not run here.')


## Task 6 — Optional spectrum-to-fingerprint learning (P2)

Do not replace the validated retrieval baseline with a neural model prematurely. This cell defines a compact spectrum encoder and Morgan-fingerprint target model (PyTorch optional); train only after group split and baseline numbers exist. Use BCE loss with logits and early stopping on molecule-disjoint validation. The initial settings below are safe starting points, **not tuned parameters**.


In [ ]:
# Compact model definition; training intentionally opt-in and requires a labeled, molecule-split dataset.
try:
    import torch
    from torch import nn
    class SpectrumFingerprintNet(nn.Module):
        def __init__(self,input_dim=4096,fp_dim=2048,width=1024,dropout=.2):
            super().__init__(); self.net=nn.Sequential(nn.Linear(input_dim,width),nn.LayerNorm(width),nn.GELU(),nn.Dropout(dropout),
                nn.Linear(width,width//2),nn.GELU(),nn.Dropout(dropout),nn.Linear(width//2,fp_dim))
        def forward(self,x):return self.net(x)
    def fit_spectrum_fingerprint(model, X_train, y_train, X_valid, y_valid, config=None, device=None):
        cfg={**LEARNING_STARTER,**(config or {})}; device=device or ('cuda' if torch.cuda.is_available() else 'cpu')
        model=model.to(device); opt=torch.optim.AdamW(model.parameters(),lr=cfg['lr'],weight_decay=cfg['weight_decay'])
        loss_fn=nn.BCEWithLogitsLoss(); best=float('inf');best_state=None;stale=0;history=[]
        tx=torch.as_tensor(X_train,dtype=torch.float32);ty=torch.as_tensor(y_train,dtype=torch.float32)
        vx=torch.as_tensor(X_valid,dtype=torch.float32,device=device);vy=torch.as_tensor(y_valid,dtype=torch.float32,device=device)
        loader=torch.utils.data.DataLoader(torch.utils.data.TensorDataset(tx,ty),batch_size=cfg['batch_size'],shuffle=True)
        for epoch in range(cfg['epochs']):
            model.train(); total=0.;n=0
            for xb,yb in loader:
                xb=xb.to(device);yb=yb.to(device);opt.zero_grad(set_to_none=True);loss=loss_fn(model(xb),yb);loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),5.);opt.step();total+=float(loss.detach())*len(xb);n+=len(xb)
            model.eval()
            with torch.no_grad(): vloss=float(loss_fn(model(vx),vy))
            history.append({'epoch':epoch+1,'train_bce':total/max(n,1),'valid_bce':vloss})
            if vloss<best:best=vloss;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};stale=0
            else:stale+=1
            if stale>=cfg['early_stopping_patience']:break
        if best_state:model.load_state_dict(best_state)
        return model,pd.DataFrame(history)
    print('Torch',torch.__version__,'; optional MLP and early-stopped fit function defined; fitting requires identity-disjoint labels.')
except ImportError:
    torch=None; SpectrumFingerprintNet=None; print('PyTorch unavailable; retrieval baseline remains usable.')

def spectrum_vector(mzs,intensities,precursor_mz,bins=4096,max_mz=1200):
    mz,it=preprocess_spectrum(mzs,intensities,precursor_mz,max_peaks=256)
    vec=np.zeros(bins,dtype=np.float32)
    if len(mz):
        idx=np.clip((mz/max_mz*bins).astype(int),0,bins-1)
        np.maximum.at(vec,idx,np.sqrt(it).astype(np.float32))
    return vec

LEARNING_STARTER={'width':1024,'dropout':0.2,'lr':3e-4,'weight_decay':1e-4,'batch_size':256,'epochs':30,'early_stopping_patience':4}
print('Starter search space:',LEARNING_STARTER)


## Task 7 — Evidence fusion and ranking (P2)

Use per-molecule, multi-spectrum evidence. Combine only calibrated channels on a group-held-out validation set. Missing channels remain missing (not zero evidence) until explicitly calibrated. The baseline helper below normalizes channel scores by rank and supports fixed weights for reproducibility; tune weights against MRR@25 and compare to each individual channel with bootstrap intervals.


In [ ]:
def fuse_ranked_channels(channel_rankings, weights=None, k=25):
    weights=weights or {name:1. for name in channel_rankings}
    combined={}
    for channel,ranking in channel_rankings.items():
        w=float(weights.get(channel,0.))
        for rank,identity in enumerate(ranking,1):combined[identity]=combined.get(identity,0.)+w/(rank**.5)
    return sorted(combined,key=lambda x:(-combined[x],x))[:k]

assert fuse_ranked_channels({'a':['x','y'],'b':['y','x']}, {'a':1,'b':1})[0] in {'x','y'}
print('Task 7 deterministic rank fusion smoke check passed; no validation gain claimed.')


## Task 8 — Offline submission assembly (P0)

Assemble exactly one row per molecule ID, deduplicate by official identity, cap at 25 candidates, validate SMILES and schema, and write `submission.csv`. For the visible preview, use library retrieval if a train pool exists; otherwise fail closed rather than inventing arbitrary molecular guesses. Hidden-test IDs are only known at the Kaggle rerun. External structure databases or pretrained weights must be permitted, documented, and available offline.


In [ ]:
def assemble_submission(test_df, candidate_map, out_path=OUTPUT_DIR/'submission.csv'):
    ids=list(dict.fromkeys(test_df.molecule_id.astype(str)))
    rows=[]
    for mid in ids:
        guesses=dedup_smiles(candidate_map.get(mid,[]),25)
        if not guesses: raise ValueError(f'No candidate for {mid}; refusing to emit invalid placeholder.')
        rows.append({'molecule_id':mid,'smiles':';'.join(guesses)})
    sub=pd.DataFrame(rows,columns=['molecule_id','smiles'])
    if sub.molecule_id.duplicated().any() or sub.isna().any().any():raise AssertionError('invalid submission schema')
    if set(sub.molecule_id)!=set(ids):raise AssertionError('submission ID coverage failure')
    for s in sub.smiles:
        parts=s.split(';')
        if len(parts)>25 or any(ik14(x) is None for x in parts):raise AssertionError('invalid candidate list')
    sub.to_csv(out_path,index=False)
    return sub

# Gate audit; no submission is generated during EDA-only or no-train runs.
print('Submission helper ready. End-to-end gate: requires train labels, molecule-disjoint benchmark, hidden-ID rerun, offline runtime <=9h.')


## Pipeline gates and next actions

| Task | Deliverable in this notebook | Current evidence / gate | Next step |
|---|---|---|---|
| 2 Identity/MRR | pinned RDKit key, dedupe, MRR helper and assertions | Runtime version must equal official pin; public examples smoke-tested | Compare against official scorer edge cases in pinned environment |
| 3 Preprocessing | aligned-array checks, deterministic top-N/threshold/binning, adduct shifts | deterministic/unit smoke tests; no filtering threshold tuned | ablate preprocessing on molecule-held-out validation |
| 4 Retrieval | exact/modified cosine and entropy-style similarity, precursor/molecule fusion | synthetic primitive checks only; train labels required | index train spectra and report Recall@K/MRR by domain |
| 5 Candidate generation | structure pool, neutral-mass filtering, identity dedupe | code path ready; public database licensing not evaluated | audit source license and measure candidate recall/coverage |
| 6 Learning | spectrum vector and compact fingerprint MLP architecture | architecture only; no fit/performance claim | train after baseline using molecule-group split, early stopping |
| 7 Ranking | deterministic channel rank fusion | smoke check only | compare against strongest channel with bootstrap uncertainty |
| 8 Submission | strict ID/list/SMILES validator and CSV writer | helper only; hidden IDs unavailable locally | Kaggle offline rerun <=9h and official scorer check |

**Do not interpret code presence as acceptance.** The currently mounted workspace lacks `train.parquet`; therefore Tasks 4–7 have no measured validation result and Task 8 has no valid competition submission. The visible test preview is training-derived.
